# Analysis Notebook for Radio Star scans using BigDish

dsheen 2025-10-02

The following is a notebook for analysis of radio star scans measured using scan_radio_star.py. Over the course of the notebook we will import the log files, perform rfi-robust processing of the measured data, and use the result to estimate calibrator curves and the telescope sensitivity over the full protected band.

The results of this analysis for a known reference star can then be used to perform wideband calibration of scans of unknown sources.

## 1. Manual Generation of Radio Star Flux Curves

In [13]:
import numpy as np
import astropy.units as u
import astropy.constants as const
import matplotlib.pyplot as plt

### Create a Nice Dictionary of Radio Stars we have Good Published Measurements for
use sources from

"An Accurate Flux Density Scale from 50MHz to 50GHz"

R. A. Perley and B. J. Butler National Radio Astronomy Observatory, P.O. Box O, Socorro, NM 87801, USA; RPerley@nrao.edu, BButler@nrao.edu

https://iopscience.iop.org/article/10.3847/1538-4365/aa6df9/pdf

so that we know we have good flux models for them. Only use relatively bright ones that are easy to find. Also save flux coefficents for them from table 6 so we can use this equation

$log(S)=a_0 +a_1log(f)+a_2[log(f)]^2 +a_3[log(f)]^3+ $.... for f in GHz and S in Jy

In [2]:
radio_stars = {}

radio_stars['3C48'] = {'coords':(24.42208192967, 33.15974455245), 
                       'coefficients' : np.array([1.3253, -0.7553,-0.1914,0.0498])}

radio_stars['FornaxA'] = {'coords':(50.6741208, -37.2082), 
                       'coefficients' : np.array([2.2175, -0.6606])}

radio_stars['3C123'] = {'coords':(69.26823064375, 29.67050498975), 
                       'coefficients' : np.array([1.8017, -0.7884, -0.1035, -0.0248, 0.0090])}

radio_stars['3C138'] = {'coords':(80.29119151075, 16.63945876275), 
                       'coefficients' : np.array([1.0088, -0.4981, -0.1552, -0.0102, 0.0223])}

radio_stars['PictorA'] = {'coords':(79.95717888494, -45.77884796187), 
                       'coefficients' : np.array([1.9380, -0.7470, -0.0739])}

radio_stars['TaurusA'] = {'coords':(83.6324, 22.0174), 
                       'coefficients' : np.array([2.9516, -0.2173, -0.0473, -0.0674]),
                    'beamwidth_correction' : 4.6}

radio_stars['3C147'] = {'coords':(85.65057457125, 49.852009364972), 
                       'coefficients' : np.array([1.4516, -0.6961, -0.2007, 0.0640, -0.0464, 0.0289])}

radio_stars['3C196'] = {'coords':(123.40023382088, 48.21740137945), 
                       'coefficients' : np.array([1.2872, -0.8530, -0.1534, -0.0200, 0.0201])}

radio_stars['HydraA'] = {'coords':(139.52361869175, -12.095501689528), 
                       'coefficients' : np.array([1.7795, -0.9176, -0.0843, -0.0139, 0.0295])}

radio_stars['VirgoA'] = {'coords':(187.70593076725, 12.391123246083), 
                       'coefficients' : np.array([2.4466, -0.8116, -0.0483]),
                         'beamwidth_correction' : 2.5}

radio_stars['3C286'] = {'coords':(202.78453479432, 30.50915558099), 
                       'coefficients' : np.array([1.2481, -0.4507, -0.1798, 0.0357])}

radio_stars['3C295'] = {'coords':(212.836, 52.2025), 
                       'coefficients' : np.array([1.4701, -0.7658, -0.2780, -0.0347, 0.0399])}

radio_stars['HerculesA'] = {'coords':(252.78328611990997, 4.99320753652), 
                       'coefficients' : np.array([1.8298, -1.0247, -0.0951])}

radio_stars['3C353'] = {'coords':(260.11736689063, -0.9796062372), 
                       'coefficients' : np.array([1.8627, -0.6938, -0.0998, -0.0732])}

radio_stars['3C380'] = {'coords':(277.38242072207, 48.74615560247), 
                       'coefficients' : np.array([1.2320, -0.7909, 0.0947, 0.0976, -0.1794, -0.1566])}

radio_stars['CygnusA'] = {'coords':(299.868152368208, 40.733915897917), 
                       'coefficients' : np.array([3.3498, -1.0022, -0.2246, 0.0227, 0.0425]),
                         'beamwidth_correction' : 2.5}

radio_stars['3C444'] = {'coords':(333.6073, -17.0267472), 
                       'coefficients' : np.array([1.1064, -1.0052, -0.0750, -0.0767])}

radio_stars['CassiopeiaA'] = {'coords':(350.8584, 58.8113), 
                        'coefficients' : np.array([3.3584, -0.7518, -0.0347, -0.0705]),
                        'beamwidth_correction' : 4.6}

radio_stars['Orion'] = {'coords':(83.8201, -5.3876), 
                       'coefficients' : np.array([2.705, -0.204]),
                       'beamwidth_correction' : 4.6}

In [3]:
def get_flux(star,f):
    '''
    star: str, name of star correcponding to dictinary entry key
    f: frequency or frequencies at which to compute flux
    '''
    
    a = radio_stars[star]['coefficients']
    logfreq = np.log10(f.to(u.GHz).value)
    
    
    logS = np.zeros_like(logfreq)
    for i in range(len(a)):
        logS += a[i]*np.power(logfreq,i)

    S = np.power(10,logS)*u.Jy
    return S

### Antenna Parameters and Corrections

from the corrections in RECOMMENDATION ITU-R S.733-2

$\frac{G}{T} = \frac{8\pi k (r-1)}{\lambda^2 \Phi(f)}$

where $r = \frac{P_n + P_{st}}{P_n} = \frac{T_{sys} + T_{st}}{T_{sys}}$ and $\Phi(f)$ is the flux in $Wm^{-2}Hz^{-1}$ 


corrrection factor C2 is included for the angular extent of radio stars vs the beam width in equation 3

$(G/ T)_c = G/T + C1 + C2 + C3$

where C3 is only applicable to Cas A, C1 is a correction for atmospheric loss (which we shall ignore because I typically count that as part of G/T) and so C2 is the only thing that matters


$C2 \approx -10log_{10}\left[\frac{ABS \left( 1-e^{-\chi^2} \right)}{\chi^2}\right]$ in dB

where $\chi_{CygA} \approx \frac{2.5}{1.2012 \theta_{3dB} \times 60}$

and $\chi_{most other things} \approx \frac{4.6}{1.2012 \theta_{3dB} \times 60}$

In [4]:
def get_C2(star,Theta):
    '''
    star is name of star in dictionary
    Theta is half power beamwidth in degrees
    freq is frequency
    
    '''
    Chi = radio_stars[star]['beamwidth_correction']/(1.2012*Theta.to(u.degree).value*60)
    C2 = -10*np.log10(np.abs(1-np.exp(-Chi**2))/Chi**2)
    return C2

## 2. Import Log Data from Scan of Radio Star

Before we can do anything with the actual RF samples, we need to know where the telescope was pointed at what time and what the calibrator states were. for this we need to ingest and process the log file since we do not yet have a mechanism for adding this to the drf metadata during the recording.

the size of the files makes it impractical to keep any in the github so you will need to get your own. the DRF top directory and the logfile should be in the same folder

In [31]:
import os
import csv
from datetime import datetime
import pandas as pd

### Get File Information

In [ ]:
#observation directory path
observation_directory = "/home/dsheen/temp_rf_data/cygnusA_2025-10-01_scan/"

#grab logfile and drf file paths
files = os.listdir(observation_directory)

for file in files:
    if file.split('.')[-1] == 'log':
        logfile = file
        print(f"found log file {logfile}")
    else: #see if it's the drf directory
        try:
            subdirs = os.listdir(os.path.join(observation_directory, file))
            if 'LHCP' in subdirs or 'RHCP' in subdirs: #then this is probably the drf top directory
                drf_directory = file
        except:
            print(f"skipping {file}. File does not appear to be digital_rf or logfile.")

found log file 2025-10-02T00:56:29.283430Z_radec_299.87_299.87.log


### Read in the log data

In [65]:
scan_target_commands = []
scan_settling_times = []
scan_calibrator_transitions = []
telescope_positions = []

with open(os.path.join(observation_directory,logfile), 'r') as file:
    logdata = csv.reader(file)

    for row in logdata:
        #splitup by info type and save into python variables and lists

        #####################
        # header info
        #####################
        if row[0].strip() == "scan_coords":
            scan_coord_frame = row[1].strip() #coordinate system
        elif row[0].strip() == "scan_center":
            scan_center = np.array([float(row[i].strip()) for i in range(1,len(row))]) #scan center point
        elif row[0].strip() == "extents":
            extents = np.array([float(row[i].strip()) for i in range(1,len(row))]) #max range of scan from center point
        elif row[0].strip() == "steps":
            steps = np.array([float(row[i].strip()) for i in range(1,len(row))]) #step between points in both axes
        elif row[0].strip() == "num_points":
            num_points = int(row[1].strip()) #number of points in scan
        elif row[0].strip() == "integration_time":
            integration_time = float(row[1].strip()) #time in seconds to integrate after calibration period
        elif row[0].strip() == "cal_time":
            cal_time = float(row[1].strip()) #time for calibrator to be enabled on each channel

        ##########################
        # actual scan timing data
        ##########################
        elif row[0].strip() == "target_point":
            #append command time and coordinates
            scan_target_commands.append({'time': datetime.fromisoformat((row[1].strip()).replace('Z', '+00:00')), 
                                         'pos' : np.array([float(row[3].strip()),float(row[4].strip())])})

        elif row[0].strip() == "position":
            #telescope reported position time, coords, velocities
            telescope_positions.append({'time': datetime.fromisoformat((row[1].strip()).replace('Z', '+00:00')), 
                                        'pos' : np.array([float(row[2].strip()),float(row[3].strip())]), 
                                        'vel' : np.array([float(row[4].strip()),float(row[5].strip())])})
        elif row[0].strip() == "settled":
            #time when telescope has settled to target point
            scan_settling_times.append(datetime.fromisoformat((row[1].strip()).replace('Z', '+00:00')))
        elif row[0].strip() == "calibrator_state":
            #calibrator state timing info
            scan_calibrator_transitions.append({'time': datetime.fromisoformat((row[1].strip()).replace('Z', '+00:00')),
                                                'state': int(row[2].strip())})
        else: #something weird happened
            print(f"unrecognized line starting with {row[0].strip()}! Is file correct?")



print(f"scan contains {num_points} points centered about {scan_coord_frame} coordinates {scan_center}")
#telescope_positions = pd.DataFrame(telescope_positions)
#print(telescope_positions)

scan contains 225 points centered about radec coordinates [299.869  40.739]


### Manipulate this into a nicer format with a signle entry for each point telling us what segments of data are what

In [82]:
scan_points_info = []

#get an array of position timestamps for subsequent use
telescope_position_times = np.array([point['time'].timestamp() for point in telescope_positions])
telescope_position_coords = np.array([point['pos'] for point in telescope_positions])

def get_pointing_time_inices(bounds):
    tmin = bounds[0].timestamp()
    tmax = bounds[1].timestamp()
    imin = np.where(telescope_position_times>=tmin)[0][0]
    imax = np.where(telescope_position_times>=tmax)[0][0]
    return imin,imax

def get_mean_pointing_error(bounds, point):
    """return telescope mean error fromm target"""
    imin,imax = get_pointing_time_inices(bounds)
    positions = telescope_position_coords[imin:imax]
    errors = np.sqrt(np.sum(np.power(positions - point,2),axis=1))
    
    return np.mean(errors)


for i in range(num_points):
    scan_target_coords = scan_target_commands[i]["pos"]
    settling_time = scan_settling_times[i]
    cal_info = scan_calibrator_transitions[3*i:3*i+3]

    if i+1 ==num_points:
        end_time = telescope_positions[-1]["time"]
    else:
        end_time = scan_target_commands[i+1]["time"]

    cal_1_bounds = [cal_info[0]['time'], cal_info[1]['time']]
    cal_2_bounds = [cal_info[1]['time'], cal_info[2]['time']]
    cal_off_bounds = [cal_info[2]['time'], end_time]

    #get nominal pointing errors for these time ranges
    
    #cal1_timestamps 

    scan_points_info.append({'pos' : scan_target_coords,
                             'cal_1_bounds' : cal_1_bounds,
                             'cal_2_bounds' : cal_2_bounds,
                             'cal_off_bounds' : cal_off_bounds,
                             'cal_1_pos_error' : get_mean_pointing_error(cal_1_bounds, scan_target_coords),
                             'cal_2_pos_error' : get_mean_pointing_error(cal_2_bounds, scan_target_coords),
                             'cal_off_pos_error' : get_mean_pointing_error(cal_off_bounds, scan_target_coords),
                            })
    
scan_points_info = pd.DataFrame(scan_points_info)

print(scan_points_info)

                   pos                                       cal_1_bounds  \
0    [294.619, 35.489]  [2025-10-02 00:56:36+00:00, 2025-10-02 00:56:3...   
1    [295.369, 35.489]  [2025-10-02 00:56:50+00:00, 2025-10-02 00:56:5...   
2    [296.119, 35.489]  [2025-10-02 00:57:04+00:00, 2025-10-02 00:57:0...   
3    [296.869, 35.489]  [2025-10-02 00:57:18+00:00, 2025-10-02 00:57:2...   
4    [297.619, 35.489]  [2025-10-02 00:57:32+00:00, 2025-10-02 00:57:3...   
..                 ...                                                ...   
220  [302.119, 45.989]  [2025-10-02 01:48:42+00:00, 2025-10-02 01:48:4...   
221  [302.869, 45.989]  [2025-10-02 01:48:58+00:00, 2025-10-02 01:49:0...   
222  [303.619, 45.989]  [2025-10-02 01:49:13+00:00, 2025-10-02 01:49:1...   
223  [304.369, 45.989]  [2025-10-02 01:49:27+00:00, 2025-10-02 01:49:2...   
224  [305.119, 45.989]  [2025-10-02 01:49:45+00:00, 2025-10-02 01:49:4...   

                                          cal_2_bounds  \
0    [2025-10-02 